In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random
import os

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
dataset_path = r"C:\Users\SURAJ CHOUGULE\Downloads\brain tumour"

train_path = dataset_path + r"\Training"
test_path = dataset_path + r"\Testing"
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    train_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    train_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_data.class_indices.keys())

print("Classes:", class_names)
images, labels = next(train_data)

plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis("off")

plt.show()
plt.imshow(images[1])
plt.title(class_names[np.argmax(labels[1])])
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

model.summary()
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    train_data,
    epochs=20,
    validation_data=val_data
)
loss, acc = model.evaluate(test_data)

print("Test Accuracy:", acc)
print("Test Loss:", loss)
test_data.reset()

pred = model.predict(test_data)
pred_classes = np.argmax(pred, axis=1)

print("Actual:", test_data.classes[:10])
print("Predicted:", pred_classes[:10])
import textwrap

test_data.reset()
images, labels = next(test_data)

plt.figure(figsize=(30, 30))

for i in range(10):
    actual = np.argmax(labels[i])
    pred = pred_classes[i]

    plt.subplot(2, 5, i+1)
    plt.imshow(images[i])

    actual_name = textwrap.fill(class_names[actual], width=14)
    pred_name = textwrap.fill(class_names[pred], width=14)

    plt.title(f"Actual: {actual_name}\nPredicted: {pred_name}",
              color='green' if actual == pred else 'red',
              fontsize=8)

    plt.axis('off')

plt.tight_layout(pad=2)
plt.show()
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.show()
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_data.reset()
y_pred = np.argmax(model.predict(test_data, verbose=0), axis=1)
y_true = test_data.classes

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)

ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    cmap='Blues', xticks_rotation=45
)
plt.show()
